In [ ]:
import yaml
import numpy as np
import polars as pl

patho_labels  = ['Pathogenic', 'Likely_pathogenic']
benign_labels = ['Benign', 'Likely_benign']

clinvar_labels = patho_labels + benign_labels

# Create input for VEP and annotation pipeline

In [ ]:
CLINVAR_URL = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar_20260621.vcf.gz"

!wget -P PATH_TO_FILE {CLINVAR_URL}


In [ ]:
annotation_dir = "PATH_TO_FILE"
clinvar_vcf_path = "PATH_TO_FILE"

clinvar = (
    pl.scan_csv(
        clinvar_vcf_path,
        separator="\t",
        comment_prefix="##",
        schema_overrides={"#CHROM": pl.Utf8, "POS": pl.Int64},
        ignore_errors=True,
    )
    .rename({"#CHROM": "chrom", "POS": "pos", "REF": "ref", "ALT": "alt"})
    .with_columns(
        chrom="chr" + pl.col("chrom").cast(pl.Utf8).str.replace(r"^chr", ""),
        clinical_significance=pl.col("INFO").str.extract(r"CLNSIG=([^;]+)", 1),
    )
    .with_columns(
        id=pl.concat_str(["chrom", "pos", "ref", "alt"], separator=":"),
    )
    .select(["id", "chrom", "pos", "ref", "alt", "clinical_significance"])
    .filter(pl.col("clinical_significance").is_in(clinvar_labels))
    .unique(subset="id")          # one row per variant
    .collect()
)

# 1. variant_metadata.parquet — exactly the schema the Snakefile consumes
clinvar.select("id", "chrom", "pos", "ref", "alt").write_parquet(
    f"{annotation_dir}/variant_metadata.parquet"
)

# 2. keep the labels to join back onto the final annotation output on `id`
clinvar.select("id", "clinical_significance").write_parquet(
    f"{annotation_dir}/clinvar_labels.parquet"
)


# Add clinical significance labels to the final annotation output

In [ ]:
annotation_dir = "PATH_TO_FILE"
cv_labels = pl.read_parquet(f"{annotation_dir}/clinvar_labels_20260621.parquet")
cv_labels

In [ ]:
cv_all = (
    pl.read_parquet("PATH_TO_FILE")
)

cv_all

In [ ]:
cv_all_labs = (
    cv_labels
    .join(
        cv_all,
        on="id",
        # how="left",
        validate="1:m"
    )
)

cv_all_labs

In [ ]:
cv_all_labs.write_parquet(
    f"{annotation_dir}/clinvar_significance_vep_annotations_processed_cadd_fill_na_20260621.parquet"
)